In [2]:
# Data manipulation
import numpy as np
import pandas as pd

# Graphs
import seaborn as sns
import matplotlib.pyplot as plt

# warnings
import warnings

In [3]:
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_style("ticks")
odx = pd.IndexSlice
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [4]:
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [5]:
def read_url(link):
    """ Creates a pandas DataFrame from data online
    - Parameters:
        - link: link to the zipped data
    - Returns:
    """
    import io
    import requests
    import pandas as pd

    # Define URL and extract information
    response = requests.get(link)
    content = response.content
    # Convert into a Pandas DataFrame
    df = pd.read_csv(io.BytesIO(content), sep=',', compression='gzip')

    return df

# calendar = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/calendar.csv.gz')
calendar = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2026-03-30/data/calendar.csv.gz')

In [6]:
calendar

,listing_id,date,available,minimum_nights,maximum_nights
0,35797,2026-04-01,f,1,7
1,35797,2026-04-02,f,1,7
2,35797,2026-04-03,t,1,7
3,35797,2026-04-04,t,1,7
4,35797,2026-04-05,t,1,7
...,...,...,...,...,...
8311074,1651930371667422569,2027-03-29,t,1,30
8311075,1651930371667422569,2027-03-30,t,1,30
8311076,1651930371667422569,2027-03-31,t,1,30
8311077,1651930371667422569,2027-04-01,t,1,30


In [7]:
# Time series analysis with calendar data
if 'date' in calendar.columns.to_list():
    calendar['date'] = pd.to_datetime(calendar['date'])

# Convert price to numeric (remove $ and ,)
if 'price' in calendar.columns.to_list():
    calendar['price'] = calendar['price'].replace('[\$,]', '', regex=True).astype(float)

print(calendar.shape)

(8311079, 5)


In [8]:
calendar['date'].min()

Timestamp('2026-03-30 00:00:00')

In [9]:
calendar_2 = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/calendar.csv.gz')


In [10]:
calendar_2['date'].max()

'2026-07-01'

In [11]:
calendar_2

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,35797,2025-06-26,f,NaN,NaN,1,7
1,35797,2025-06-27,t,NaN,NaN,1,7
2,35797,2025-06-28,t,NaN,NaN,1,7
3,35797,2025-06-29,t,NaN,NaN,1,7
4,35797,2025-06-30,t,NaN,NaN,1,7
...,...,...,...,...,...,...,...
9636360,1446435219088525505,2026-06-27,f,NaN,NaN,1,365
9636361,1446435219088525505,2026-06-28,f,NaN,NaN,1,365
9636362,1446435219088525505,2026-06-29,f,NaN,NaN,1,365
9636363,1446435219088525505,2026-06-30,f,NaN,NaN,1,365


In [12]:
columns = calendar_2.columns.to_list()

columns_names = ", ".join(columns)
placeholders = ", ".join(["%s"] * len(columns))

update_clause = ", ".join(
    [   
        f"{col} = EXCLUDED.{col}"
        for col in columns
        if col not in ["id", "date"]
    ]
)

In [13]:
columns

['listing_id',
 'date',
 'available',
 'price',
 'adjusted_price',
 'minimum_nights',
 'maximum_nights']

In [14]:
columns_names

'listing_id, date, available, price, adjusted_price, minimum_nights, maximum_nights'

In [15]:
placeholders

'%s, %s, %s, %s, %s, %s, %s'

In [16]:
update_clause

'listing_id = EXCLUDED.listing_id, available = EXCLUDED.available, price = EXCLUDED.price, adjusted_price = EXCLUDED.adjusted_price, minimum_nights = EXCLUDED.minimum_nights, maximum_nights = EXCLUDED.maximum_nights'

In [17]:
query = f"""
    INSERT INTO calendar ({columns_names})
    VALUES ({placeholders})
    ON CONFLICT (id, date)
    DO UPDATE SET {update_clause}"""

In [18]:
query

'\n    INSERT INTO calendar (listing_id, date, available, price, adjusted_price, minimum_nights, maximum_nights)\n    VALUES (%s, %s, %s, %s, %s, %s, %s)\n    ON CONFLICT (id, date)\n    DO UPDATE SET listing_id = EXCLUDED.listing_id, available = EXCLUDED.available, price = EXCLUDED.price, adjusted_price = EXCLUDED.adjusted_price, minimum_nights = EXCLUDED.minimum_nights, maximum_nights = EXCLUDED.maximum_nights'